In [ ]:
"""
==============================================================================
🚀 [Step 3] Letterbox Normalization & BBox Precision Clipping
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 원본 이미지의 종횡비(Aspect Ratio) 왜곡을 방지하기 위한
###                 Letterbox 리사이징 및 모델 입력 규격(800x800) 표준화.
###                 이미지 경계선 밖으로 이탈하는 BBox 좌표에 대한 정밀 클리핑 로직 구현.
### @Key Metric   : Resolution 800x800 (Padding: 114,114,114), Zero-Data-Loss Clipping
### @Input        : train_augmented_final.json, val.json
### @Output       : train_letterbox.json, val_letterbox.json, letterbox_images/
==============================================================================
"""


In [ ]:
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass  # 이미 마운트됐거나 VSCode 커널 환경

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    # nested 폴더 대응 (pill_detection_project/pill_detection_project/src/ 구조)
    PROJECT_ROOT = REPO_DIR
    nested = os.path.join(REPO_DIR, 'pill_detection_project')
    if os.path.isdir(os.path.join(nested, 'src')):
        PROJECT_ROOT = nested

    sys.path.insert(0, PROJECT_ROOT)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    REPO_DIR = os.path.expanduser('~/Desktop/vera_pill')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = os.path.expanduser('~/Desktop/vera_pill/dataset')

print(f"환경    : {'Colab' if is_colab else '로컬'}")
print(f"REPO_DIR: {sys.path[0]}")
print(f"BASE_DIR: {BASE_DIR}")
assert os.path.exists(BASE_DIR), f"🚨 경로 없음: {BASE_DIR}"


In [ ]:
import os, json, glob, warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm
from src.preprocessing.transforms import run_letterbox_pipeline
from src.preprocessing.viz_utils import show_samples, show_letterbox_comparison

plt.rcParams['font.family']        = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
print("✅ 라이브러리 로드 완료")


## 1. 경로 설정

In [ ]:
TARGET_SIZE = 800  # YOLO, Faster R-CNN 등 대부분의 모델이 선호하는 고해상도 규격

# [Input] 증강이 완료된 Train 데이터와 분리된 Val 데이터
AUG_JSON_PATH = os.path.join(BASE_DIR, 'train_augmented_final.json')
VAL_JSON_PATH = os.path.join(BASE_DIR, 'val.json')

# [Output] 모델이 바로 소화할 수 있는 최종 데이터 경로
LB_TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'letterbox_images/train')
LB_VAL_IMG_DIR   = os.path.join(BASE_DIR, 'letterbox_images/val')
LB_TRAIN_JSON    = os.path.join(BASE_DIR, 'train_letterbox.json')
LB_VAL_JSON      = os.path.join(BASE_DIR, 'val_letterbox.json')

assert os.path.exists(AUG_JSON_PATH), f"없음: {AUG_JSON_PATH} — NB02를 먼저 실행하세요"
assert os.path.exists(VAL_JSON_PATH), f"없음: {VAL_JSON_PATH}"
print("✅ 경로 확인 완료")


## 2. Train Letterbox 변환

In [ ]:
print(f"📦 [Phase 4] Letterbox 데이터 가공 파이프라인을 시작합니다.\n")

run_letterbox_pipeline(
    json_path     = AUG_JSON_PATH,
    out_json_path = LB_TRAIN_JSON,
    img_out_dir   = LB_TRAIN_IMG_DIR,
    base_dir      = BASE_DIR,
    target_size   = TARGET_SIZE,
    desc          = 'Train Letterbox 변환',
)


## 3. Val Letterbox 변환

In [ ]:
run_letterbox_pipeline(
    json_path     = VAL_JSON_PATH,
    out_json_path = LB_VAL_JSON,
    img_out_dir   = LB_VAL_IMG_DIR,
    base_dir      = BASE_DIR,
    target_size   = TARGET_SIZE,
    desc          = 'Val Letterbox 변환',
)

print("\n✨ [엔지니어링 완료] 모델링 투입 준비가 모두 끝났습니다!")
print(f"  - 🎯 Train 정답지: {os.path.basename(LB_TRAIN_JSON)}")
print(f"  - 🧪 Val 정답지  : {os.path.basename(LB_VAL_JSON)}")


## 4. 📸 Letterbox 전후 비교 (5장)

In [ ]:
show_letterbox_comparison(
    orig_img_dir = os.path.join(BASE_DIR, 'train_images'),
    lb_img_dir   = LB_TRAIN_IMG_DIR,
    orig_json    = os.path.join(BASE_DIR, 'merged_annotations_train_final.json'),
    lb_json      = LB_TRAIN_JSON,
    n            = 5,
)


## 5. 📸 Train Letterbox 결과 20장

In [ ]:
show_samples(LB_TRAIN_IMG_DIR, LB_TRAIN_JSON, n=20, title="Train Letterbox 결과 (800×800)")


## 6. 📸 Val Letterbox 결과 20장

In [ ]:
show_samples(LB_VAL_IMG_DIR, LB_VAL_JSON, n=20, title="Val Letterbox 결과 (800×800)")


## 7. 이미지 크기 검증

In [ ]:
train_imgs = glob.glob(os.path.join(LB_TRAIN_IMG_DIR, '*.jpg'))
sizes = []
for p in train_imgs[:100]:  # 100장 샘플 체크
    img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_COLOR)
    if img is not None:
        sizes.append(img.shape[:2])

unique_sizes = set(sizes)
print(f"전체 고유 크기: {unique_sizes}")
assert unique_sizes == {(800, 800)}, "⚠️  800×800이 아닌 이미지가 있습니다!"
print(f"✅ 모든 이미지 800×800 확인 완료 (train {len(train_imgs)}장)")


## ✅ NB03 완료
> 다음 단계: NB04 - CLAHE 대비 강화